In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics import accuracy_score, roc_auc_score

In [2]:
# Real Dataset
real_df = pd.read_csv("test_data/adult.csv")

# Synthetic Datasets
ctgan_df = pd.read_csv("synthetic_data/ctgan_synthetic.csv")
tvae_df = pd.read_csv("synthetic_data/tvae_synthetic.csv")
# fairgan_df = pd.read_csv("synthetic_data/fairgan_synthetic.csv")
tabfairgan_df = pd.read_csv("synthetic_data/tabfairgan_synthetic.csv")

synthetic_datasets = {
    "CTGAN": ctgan_df,
    "TVAE": tvae_df,
    "TabFairGAN": tabfairgan_df
}

#Target Column
target_col = "income"

#Sensitive Column
sensitive_col= "sex"

# Downstream Classifier

## Preprocessing

In [3]:
real_train_df, real_test_df = train_test_split(real_df, test_size=0.2, random_state=42)

def encode_target(df, target_col):
    """
    Encodes the target column into binary
    """
    df = df.copy()
    df[target_col] = df[target_col].astype(str)
    df[target_col] = df[target_col].str.strip()
    df[target_col] = df[target_col].str.replace(".", "", regex=False)
    df[target_col] = df[target_col].map({
        "<=50K": 0,
        ">50K": 1
    })
    return df

real_train_df = encode_target(real_train_df, "income")
real_test_df = encode_target(real_test_df, "income")

ctgan_df = encode_target(ctgan_df, "income")
tvae_df = encode_target(tvae_df, "income")
# fairgan_df = encode_target(fairgan_df, "income")
tabfairgan_df = encode_target(tabfairgan_df, "income")


def build_pipeline(df, target_col):
    """
    Builds a preprocessing + Logistic Regression pipeline
    based on the structure of the given dataset.
    """

    X = df.drop(columns=[target_col])
    
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("num", "passthrough", numerical_cols)
        ]
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ])
    
    return pipeline

### Train the classifier

In [4]:
def train_pipeline(train_df, target_col):
    """
    Trains a pipeline on a given dataset (real or synthetic).
    """
    
    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]

    pipeline = build_pipeline(train_df, target_col)
    pipeline.fit(X_train, y_train)

    return pipeline

### Run the training on the real dataset and the synthetic dataset 

In [5]:
# Train on the real dataset
real_model = train_pipeline(real_train_df, target_col)

# Train on synthetic dataseyts
synthetic_datasets = {
    "CTGAN": ctgan_df,
    "TVAE": tvae_df,
    "TabFairGAN": tabfairgan_df
}
synthetic_models = {}

for name, df in synthetic_datasets.items():
    synthetic_models[name] = train_pipeline(df, target_col)

/Users/staceykagio/fair-synthetic-data-gen/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/staceykagio/fair-synthetic-data-gen/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    h

# Evaluation

## Fairness Metrics

In [6]:
# Fairness Metrics Definition
def demographic_parity(y_pred, sensitive):
    groups = sorted(sensitive.unique()) # They are sorted since order is not guaranteed i.e. results may flip
    rates = [y_pred[sensitive == g].mean() for g in groups]
    return abs(rates[0] - rates[1])


def disparate_impact(y_pred, sensitive):
    groups = sorted(sensitive.unique())
    rate_0 = y_pred[sensitive == groups[0]].mean()
    rate_1 = y_pred[sensitive == groups[1]].mean()
    return min(rate_0 / (rate_1 + 1e-6), rate_1 / (rate_0 + 1e-6))


def equal_opportunity(y_true, y_pred, sensitive):
    groups = sorted(sensitive.unique())
    tprs = []
    
    for g in groups:
        mask = (sensitive == g)
        tp = ((y_pred == 1) & (y_true == 1) & mask).sum()
        fn = ((y_pred == 0) & (y_true == 1) & mask).sum()
        tprs.append(tp / (tp + fn + 1e-6))
    
    return abs(tprs[0] - tprs[1])


def equalized_odds(y_true, y_pred, sensitive):
    groups = sorted(sensitive.unique())
    tpr_list, fpr_list = [], []
    
    for g in groups:
        mask = (sensitive == g)
        
        tp = ((y_pred == 1) & (y_true == 1) & mask).sum()
        fn = ((y_pred == 0) & (y_true == 1) & mask).sum()
        fp = ((y_pred == 1) & (y_true == 0) & mask).sum()
        tn = ((y_pred == 0) & (y_true == 0) & mask).sum()
        
        tpr = tp / (tp + fn + 1e-6)
        fpr = fp / (fp + tn + 1e-6)
        
        tpr_list.append(tpr)
        fpr_list.append(fpr)
    
    return abs(tpr_list[0] - tpr_list[1]) + abs(fpr_list[0] - fpr_list[1])


# Fairness Metrics Evaluation
def evaluate_fairness(model, test_df, target_col, sensitive_col):
    """
    Evaluates fairness metrics on the real test set.
    """
    
    # Split test data
    X_test = test_df.drop(columns=[target_col])
    y_true = test_df[target_col]
    sensitive = test_df[sensitive_col]

    # Predict using trained pipeline
    y_pred = model.predict(X_test)

    # Compute fairness metrics
    results = {
        "demographic_parity": demographic_parity(y_pred, sensitive),
        "disparate_impact": disparate_impact(y_pred, sensitive),
        "equal_opportunity": equal_opportunity(y_true, y_pred, sensitive),
        "equalized_odds": equalized_odds(y_true, y_pred, sensitive)
    }

    return results

# Comparison of Fairness Across Datasets
fairness_results = {} # Store results

# Real dataset baseline
fairness_results["Real"] = evaluate_fairness(
    real_model,
    real_test_df,
    target_col="income",
    sensitive_col="sex"   # change if needed
)

# Synthetic models
for name, model in synthetic_models.items():
    fairness_results[name] = evaluate_fairness(
        model,
        real_test_df,
        target_col="income",
        sensitive_col="sex"
    )

fairness_df = pd.DataFrame(fairness_results).T
fairness_df

,demographic_parity,disparate_impact,equal_opportunity,equalized_odds
Real,0.169725,0.282455,0.119027,0.192185
CTGAN,0.068207,0.468016,0.026791,0.045483
TVAE,0.186848,0.451374,0.072169,0.177463
TabFairGAN,0.251015,0.020518,0.516920,0.641259


## Synthetic Data Realism Metric

In [9]:
# Preprocessing for MMD 
def preprocess_mmd(df, target_col):
    """
    Converts dataframe into numeric matrix for MMD:
    - Drops target column
    - One-hot encodes categorical features
    """
    X = df.drop(columns=[target_col])
    X = pd.get_dummies(X, drop_first=True)
    return X


# MMD Computation 
def compute_mmd(X_real, X_synth, gamma=1.0):
    """
    Computes Maximum Mean Discrepancy using RBF kernel.
    """
    K_xx = rbf_kernel(X_real, X_real, gamma=gamma)
    K_yy = rbf_kernel(X_synth, X_synth, gamma=gamma)
    K_xy = rbf_kernel(X_real, X_synth, gamma=gamma)
    
    mmd = K_xx.mean() + K_yy.mean() - 2 * K_xy.mean()
    
    return mmd


# Main Evaluation Function
def evaluate_realism(real_df, synth_df, target_col, sample_size=1000):
    """
    Computes MMD using random sampling to avoid memory crash.
    """
    X_real = preprocess_mmd(real_df, target_col)
    X_synth = preprocess_mmd(synth_df, target_col)

    # Align columns
    X_synth = X_synth.reindex(columns=X_real.columns, fill_value=0)

    # --- SAMPLE (CRITICAL FIX) ---
    X_real_sample = X_real.sample(n=min(sample_size, len(X_real)), random_state=42)
    X_synth_sample = X_synth.sample(n=min(sample_size, len(X_synth)), random_state=42)

    return compute_mmd(X_real_sample.values, X_synth_sample.values)


# Run for All Synthetic Datasets
realism_results = {}

synthetic_datasets = {
    "CTGAN": ctgan_df,
    "TVAE": tvae_df,
    "TabFairGAN": tabfairgan_df
}

for name, synth_df in synthetic_datasets.items():
    realism_results[name] = evaluate_realism(
        real_train_df,
        synth_df,
        target_col="income",
        sample_size=1000 
    )

realism_df = pd.DataFrame.from_dict(
    realism_results,
    orient="index",
    columns=["MMD"]
)
realism_df

,MMD
CTGAN,0.002001
TVAE,0.002001
TabFairGAN,0.002002


## Utility Metric

In [11]:
def evaluate_utility(model, test_df, target_col):
    """
    Evaluates predictive performance using TSTR:
    Train on (real/synthetic) → Test on REAL test data
    """
    X_test = test_df.drop(columns=[target_col])
    y_true = test_df[target_col]
    y_pred = model.predict(X_test)
    
    # Probabilities (needed for AUC)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred# fallback (rare case)

    results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob)
    }

    return results

utility_results = {}

# Real dataset baseline
utility_results["Real"] = evaluate_utility(
    real_model,
    real_test_df,
    target_col="income"
)

# Synthetic models
for name, model in synthetic_models.items():
    utility_results[name] = evaluate_utility(
        model,
        real_test_df,
        target_col="income"
    )

utility_df = pd.DataFrame(utility_results).T
utility_df

,accuracy,auc
Real,0.843689,0.882280
CTGAN,0.812775,0.846902
TVAE,0.788310,0.838785
TabFairGAN,0.814822,0.840669


# Evaluation Comparison

In [13]:
final_df = utility_df.join(fairness_df)
final_df = final_df.join(realism_df)
final_df["MMD"] = final_df["MMD"].fillna('-')  # or use None or 0 if you prefer

final_df = final_df[
    [
        "accuracy",
        "auc",
        "demographic_parity",
        "disparate_impact",
        "equal_opportunity",
        "equalized_odds",
        "MMD"
    ]
]

final_df

,accuracy,auc,demographic_parity,disparate_impact,equal_opportunity,equalized_odds,MMD
Real,0.843689,0.882280,0.169725,0.282455,0.119027,0.192185,-
CTGAN,0.812775,0.846902,0.068207,0.468016,0.026791,0.045483,0.002001
TVAE,0.788310,0.838785,0.186848,0.451374,0.072169,0.177463,0.002001
TabFairGAN,0.814822,0.840669,0.251015,0.020518,0.516920,0.641259,0.002002
